<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/SentenceTransformers/notebooks/2_topic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic modeling with text embeddings and semantic similarity

# Select a GPU runtime

Before you begin, open the **Runtime** menu, select **Change runtime type** and select **T4 GPU**.

This is done because methods using embeddings run much faster on a GPU than on a CPU.

In [ ]:
# Check if you are connected to a GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# Preliminaries

In [ ]:
# import the necessary libraries
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
tqdm.pandas()
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from matplotlib import pyplot as plt
from umap import UMAP

In [ ]:
# load Sentence Transformers model (HF_TOKEN required)
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
model = SentenceTransformer('google/embeddinggemma-300m', token=HF_TOKEN)
model

# Loading the Dataset

In [ ]:
# load the dataset from GitHub
url = 'https://raw.githubusercontent.com/denisejroth/bags-vectors-transformers/main/SentenceTransformers/notebooks/bills.csv'
df = pd.read_csv(url)
df

In [ ]:
# embedding the text column
embeddings = model.encode(df['text'].to_list(), batch_size=32, show_progress_bar=True)
df['emb'] = list(embeddings)
df

# Using text embeddings to identify typical tokens

As a first step, we are going to use text embeddings to extract typical tokens from text, in a way similar to keyword extraction. This works by splitting the text into tokens (can be words or unigrams, but also bigrams or trigrams), embedding those tokens, and then comparing the cosine similarity of the tokens to a text or a list of texts.

## create stopword list

For extracting tokens we need a stopword list with tokens to be excluded, because stopwords are so frequent they would mess up the result.


In [ ]:
# Download stopword list
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stopwordlist = stopwords.words('english')

In [ ]:
# add additional stopwords if needed
stopwordlist = stopwordlist + ['united', 'states', 'amend']

## extract most frequent tokens from text column

In [ ]:
# create a function to extract the most *frequent* words from a DataFrame column with text data

from sklearn.feature_extraction.text import CountVectorizer

def get_most_frequent_tokens(df, textcol='text', ngrams=1, top_n=5, stopwordlist=stopwordlist):
  vectorizer = CountVectorizer(ngram_range=(ngrams, ngrams), stop_words=stopwordlist)
  token_matrix = vectorizer.fit_transform(df[textcol])
  tokens = vectorizer.get_feature_names_out()
  token_count_vector_df = pd.DataFrame(token_matrix.toarray(), columns=tokens)
  token_count_df = token_count_vector_df.sum().sort_values(ascending=False)
  toptokens = token_count_df.head(top_n).index.to_list()
  return toptokens


In [ ]:
# try out on the whole dataset
get_most_frequent_tokens(df, 'text', ngrams=1, top_n=10, stopwordlist=stopwordlist)

In [ ]:
# let's look at the most frequent bigrams
get_most_frequent_tokens(df, 'text', ngrams=2, top_n=10, stopwordlist=stopwordlist)

In [ ]:
# and the most frequent trigrams
get_most_frequent_tokens(df, 'text', ngrams=3, top_n=10, stopwordlist=stopwordlist)

## extract most typical tokens from text column

Here we use the Sentence Transformer model to identify the tokens that are most semantically similar to the average text in the column, from a candidate list of the 1000 most frequent tokens.

In [ ]:
# create a function to extract the most typical tokens
def get_most_typical_tokens(df, textcol='text', embcol='emb', ngrams=2, top_n=5, stopwordlist=stopwordlist):

  toptokens = get_most_frequent_tokens(df, textcol, ngrams=ngrams, top_n=1000, stopwordlist=stopwordlist)
  tokens_emb = model.encode(toptokens) #show_progress_bar=True)

  mean_doc_emb = df[embcol].mean().reshape(1, -1)

  token_doc_sim_matrix = cosine_similarity(tokens_emb, mean_doc_emb)[:,0]
  token_doc_sim_df = pd.DataFrame(token_doc_sim_matrix, index=toptokens, columns=['similarity']).sort_values(by='similarity', ascending=False)
  most_typical_tokens = token_doc_sim_df.head(top_n).index.to_list()

  return most_typical_tokens

In [ ]:
# let's try out on the whole dataset
get_most_typical_tokens(df, textcol='text', embcol='emb', ngrams=1, top_n=10, stopwordlist=stopwordlist)

In [ ]:
# most typical bigrams
get_most_typical_tokens(df, textcol='text', embcol='emb', ngrams=2, top_n=10, stopwordlist=stopwordlist)

In [ ]:
# most typical trigrams
get_most_typical_tokens(df, textcol='text', embcol='emb', ngrams=3, top_n=10, stopwordlist=stopwordlist)

## compare most frequent and most typical tokens for groups of rows in the text column

In [ ]:
# extract the most frequent bigrams and trigrams for the texts in each policy_area
group_col = 'policy_area'
groupdf = pd.DataFrame(index=sorted(df[group_col].unique()))
groupdf['frequent_tokens_2'] = df.groupby(group_col)[['text', 'emb']].progress_apply(get_most_frequent_tokens, ngrams=2)
groupdf['frequent_tokens_3'] = df.groupby(group_col)[['text', 'emb']].progress_apply(get_most_frequent_tokens, ngrams=3)
groupdf['typical_tokens_2'] = df.groupby(group_col)[['text', 'emb']].progress_apply(get_most_typical_tokens, ngrams=2)
groupdf['typical_tokens_3'] = df.groupby(group_col)[['text', 'emb']].progress_apply(get_most_typical_tokens, ngrams=3)
groupdf

# Topic modeling through clustering text embeddings

Here we use Sentence Transformer embeddings as the basis for topic modeling, in 3 steps:
1. The 768-dimensional text embeddings are reduced to 3-dimensional text embeddings using UMAP, as a preparatory step for clustering
2. The 3D text embeddings are clustered using kMeans, such that each text is assigned to one cluster
3. For each cluster, the most frequent and most typical tokens are extracted, to label the clusters
4. Each cluster is qualitatively interpreted as a topic, using the extracted tokens and the typical documents for each cluster

## Reducing the embeddings to 3 dimensions using UMAP

In [ ]:
# reduce the text embeddings to 3 dimensions for clustering purposes
umapper3D = UMAP(n_neighbors=15, n_components=3, min_dist=0.0, metric='cosine',
                 unique=True, verbose=True, random_state=42)
umap_embeddings3D = umapper3D.fit_transform(embeddings)
umap_embeddings3D.shape

In [ ]:
# store 3D coordinates in the DataFrame
df[['x3D', 'y3D', 'z3D']] = umap_embeddings3D

In [ ]:
# Create a 3D scatter plot of the text embeddings
import plotly.express as px
fig = px.scatter_3d(df, x='x3D', y='y3D', z='z3D', opacity=0.25, title='3D UMAP Embeddings')
fig.update_traces(marker_size=3)  # Set a fixed size for all markers here
fig.update_layout(width=1000, height=800) # Set figure size
fig.show()

## Clustering the 3D embeddings using Kmeans

In [ ]:
# run Kmeans clustering multiple times to find clustering solution with high silhouette score
from sklearn.cluster import KMeans

from sklearn.metrics import silhouette_score
n_clusters_list = []
silhouette_metrics = []
for n_clusters in tqdm(range(2,64)):
  clusterer = KMeans(n_clusters=n_clusters, random_state=42)
  cluster_labels = clusterer.fit_predict(umap_embeddings3D)
  n_clusters_list.append(n_clusters)
  silhouette_metrics.append(silhouette_score(umap_embeddings3D, cluster_labels))

In [ ]:
# visualize silhouette score per clustering solution (from 2 t0 64 clusters)
silhouette_Series = pd.Series(silhouette_metrics, index=n_clusters_list)
silhouette_Series.plot(grid=True, marker='o', figsize=(16,4))

In [ ]:
# select solution with maximum silhouette score within a certain range
k = silhouette_Series.loc[4:16].idxmax()
print('the solution with', k, 'clusters has a silhouette score of', silhouette_Series[k])

In [ ]:
# cluster the embeddings for the chosen number of clusters (k)
from sklearn.cluster import KMeans
clusterer = KMeans(n_clusters=k, random_state=42)
cluster_labels = clusterer.fit_predict(umap_embeddings3D)
print(cluster_labels)

In [ ]:
# store the clusters labels in the DataFrame
df['cluster'] = cluster_labels
df.cluster.value_counts().plot.bar()

In [ ]:
# Create a 3D scatter plot of the clusters of text embeddings
fig = px.scatter_3d(df, x='x3D', y='y3D', z='z3D', color='cluster', opacity=0.25, title='3D UMAP Embeddings with clusters')
fig.update_traces(marker_size=2)  # Set a fixed size for all markers here
fig.update_layout(width=1000, height=800) # Increase figure size
fig.show()

## Labeling the clusters as topics based on most frequent and most typical tokens

In [ ]:
# extract most frequent and most typical bigrams and trigrams for each cluster
topicsdf = pd.DataFrame(range(0,k), columns=['cluster'])
topicsdf['most_frequent_tokens_2'] = df.groupby('cluster')[['text', 'emb']].progress_apply(get_most_frequent_tokens, textcol='text', ngrams=2)
topicsdf['typical_tokens_2'] = df.groupby('cluster')[['text', 'emb']].progress_apply(get_most_typical_tokens, textcol='text', embcol='emb', ngrams=2)
topicsdf['most_frequent_tokens_3'] = df.groupby('cluster')[['text', 'emb']].progress_apply(get_most_frequent_tokens, textcol='text', ngrams=3)
topicsdf['typical_tokens_3'] = df.groupby('cluster')[['text', 'emb']].progress_apply(get_most_typical_tokens, textcol='text', embcol='emb', ngrams=3)
topicsdf

In [ ]:
# create a topic label for plotting
topicsdf['topiclabel'] = topicsdf['typical_tokens_2'].str.slice(0, 3).str.join(',\n')
topicsdf['topiclabel'] = 'T' + topicsdf['cluster'].astype(str).str.zfill(2) + '. ' + topicsdf['topiclabel']
topicsdf[['cluster','topiclabel']]

In [ ]:
# create topic-label dictionary
topiclabeldict = topicsdf.set_index('cluster')['topiclabel'].to_dict()
topiclabeldict

In [ ]:
# store topic labels in the DataFrame
df['topic'] = df['cluster'].map(topiclabeldict)

In [ ]:
# Create a 3D scatter plot of text embeddings, clusters and topics
fig = px.scatter_3d(df, x='x3D', y='y3D', z='z3D', color='cluster', hover_name='topic',
                    opacity=0.25, title='3D UMAP Embeddings with clusters and topic labels')
fig.update_traces(marker_size=2)  # Set a fixed size for all markers here
fig.update_layout(width=1000, height=800) # Increase figure size
fig.show()

## Visualizing the documents and topics in a 2D plot

For reporting the results, we often need a simpler 2D visualization of the texts and topics

In [ ]:
umapper2D = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine',
                 unique=True, verbose=True, random_state=42)
umap_embeddings2D = umapper2D.fit_transform(embeddings)
umap_embeddings2D.shape

In [ ]:
df[['x2D', 'y2D']] = umap_embeddings2D

In [ ]:
ax = df.plot.scatter('x2D', 'y2D', c='cluster', cmap='coolwarm', alpha=0.25, figsize=(24,16),
                     title='Two-dimensional representation of text documents and topics')
for idx, row in df.groupby('topic')[['x2D', 'y2D']].mean().iterrows():
  ax.annotate(idx, (row['x2D'], row['y2D']), fontsize=10, color='black')

## Inspecting typical documents for each topic

Here we use the text embeddings and cosine similarity again, but now to identify those text documents that are most semantically similar to the average text embedding of each topic.

In [ ]:
# create a function to identify the most typical documents in a text column
def get_typical_documents(df, textcol='text', embcol='emb', top_n=5):
  mean_doc_emb = df[embcol].mean().reshape(1, -1)
  doc_meandoc_sim_matrix = cosine_similarity(df[embcol].to_list(), mean_doc_emb)
  doc_meandoc_sim_df = pd.DataFrame(doc_meandoc_sim_matrix, index=df.index, columns=['similarity'])
  doc_meandoc_sim_df = doc_meandoc_sim_df.sort_values(by='similarity', ascending=False)
  most_typical_docs_idx = doc_meandoc_sim_df.head(top_n).index.to_list()
  most_typical_docs = df.loc[most_typical_docs_idx, textcol].to_list()
  return most_typical_docs

In [ ]:
# extract most typical documens per topic
typical_docs_df = df.groupby('topic')[['text', 'emb']].progress_apply(get_typical_documents)
typical_docs_df = pd.DataFrame(typical_docs_df, columns=['text']).reset_index()
typical_docs_df = typical_docs_df.explode('text')
typical_docs_df

# Compare topics with labeled policy areas

In [ ]:
# compare topics with labeled policy areas
pd.crosstab(df.topic, df.policy_area).style.background_gradient(cmap='Blues')

> **✏️ Exercise**


> Try a clustering solution with a smaller and higher number of clusters and see what topics come out. Clustering metrics are not the only important thing for topic modeling, interpretability of the topics is equally important!